# Load ward data using Python scripts

This uses extract and load methods in the `src/etl/csv.py` file to load ONS Ward to Local Authority District data into DuckDB.

## Background
Some of the Arts Council data (e.g. grants) uses Local Authority (LA) names whilst other data reports at the ward level.

Given this, it'd  be useful to get a ward to local authority mapping from an authoritative source. This notebook does that and also demos how to use the generic extract and load methods in the Python scripts in this project.

## Approach
This notebook uses the Python scripts in the `src` folder of the project. There are instructions at the bottom of the `readme.md` of how to create a virtual environment and install the project so that it can be imported as below. They are copied here but check the `readme` in case they have been updated since the notebook

### (Copied from readme.md) Installation

If you haven't set up a `venv` with the Python scripts in (see below) *you'll need to do this before you run this notebook*

* Requires Python 3.12 and pip
* Dependencies and other stuff are defined in the `pyproject.toml`
* From the `brighton_creatives` directory:
  * `python -m venv venv`
  * `source venv/bin/activate` (`.\venv\Scripts\activate.bat` from Windows cmd)
  * `pip install -e .[dev]`
   

* Now relaunch Jupyter Notebook from Terminal (or Windows cmd) from the `venv` you just activated. This will mean the project (including the ETL scripts) have been loaded

`jupyter notebook`


### The loader
The `load_from_template` method is a generic method that extracts data from simple CSVs and loads into DuckDb. It uses a JSON template that defines the table name, new column names, and any non-`VARCHAR` columns for the table.

In [ ]:
from etl.csv import load_from_template
import duckdb
import pandas as pd
from pathlib import Path

### Paths to the source data and templates


In [ ]:
DATA_PATH = '../data/'
PATH_TO_DB = DATA_PATH+'creatives.duckdb'
PATH_TO_ONS_WARD_DATA = DATA_PATH+'csv/ons/ward_to_local_authority_may_2025.csv'
PATH_TO_ONS_WARD_TEMPLATE = DATA_PATH+'etl_templates/ons_ward_to_lad_template.json'

### This is what the Ward to Local Authority Mapping data looks like ... 

In [ ]:
df_ward = pd.read_csv(PATH_TO_ONS_WARD_DATA, header=0, encoding='utf-8')
display(df_ward.head())

The `WD25CD` and `LAD25CD` are the ONS unique identifiers for the particular Ward and Local Authority.

The fields that end in `W` e.g. `WD25NMW` are Welsh names where appropriate, for example

In [ ]:
display(df_ward[df_ward['WD25NM'] == 'Sketty'])

### The JSON templates are used to map source columns to target (db columns) and to define the table name ...

In [ ]:
!head -n30 ../data/etl_templates/ons_ward_to_lad_template.json


The JSON also supports:
  *  `header_line` - which is a zero-indexed row where the column names are (defaults to 0)
  *  `skip_rows` - if there are non-header rows that have to be skipped
  *  `column_mappings` - if there are non-`VARCHAR columns`

For example:
```json
{
  "name": "ons-ward-to-la",
  "target_table": "ward_to_lad",
  "column_mappings": {
    "WD25CD": "ward_code",
    "WD25NM": "ward_name",
    "WD25NMW": "welsh_ward_name",
    "LAD25CD": "local_authority_code",
    "LAD25NM": "local_authority_name",
    "LAD25NMW": "welsh_local_authority_name",
    "ObjectId": "ons_internal_id"
  },
  "column_type_overrides": {
    "ons_internal_id": "TINYINT"
  }
}

```

## Loading the data
This is as simple as passing a db connection, the filepath of the source data, and the filepath of the template to `load_from_template`.


In [ ]:
with duckdb.connect(database=PATH_TO_DB, read_only=False) as con:
    con.execute("DROP TABLE ward_to_lad")
    load_from_template(con, PATH_TO_ONS_WARD_DATA, PATH_TO_ONS_WARD_TEMPLATE)
    display(con.execute("SELECT * FROM ward_to_lad LIMIT 20").df())
    display(con.execute("SELECT * FROM ward_to_lad WHERE welsh_ward_name = 'Sgeti'").df())
